# ಪಾಠ 10 - ಉತ್ಪಾದನೆಯಲ್ಲಿ AI ಏಜೆಂಟ್ಗಳು

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿಕೊಂಡು AI ಏಜೆಂಟ್ಗಳಿಗಾಗಿ **ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು** ಅನ್ನು ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಈ ವಿಷಯಗಳನ್ನು ಒಳಗೊಂಡಿದ್ದೇವೆ:

- **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಪರಸ್ಪರ ಕ್ರಿಯೆಗಳಿಗೆ ಸಮಯಮಾಪನ ಮತ್ತು ಲಾಗಿಂಗ್ ಸೇರಿಸುವುದು
- **ಮೌಲ್ಯಮಾಪನ** — ಪ್ರತಿಕ್ರಿಯೆಯ ಗುಣಮಟ್ಟಕ್ಕೆ ಅಂಕ ನೀಡಿ ಮೌಲ್ಯಮಾಪನ ಮಾಡಲು ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ಬಳಸುವುದು
- **ಖರ್ಚು ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಅನुकूलನೆ ಮತ್ತು ಮಾದರಿ ಆಯ್ಕೆಗಾಗಿ ತಂತ್ರಗಳು

ಈ ದೃಶ್ಯಾವಳಿ ಒಂದು **ಪ್ರವಾಸ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಅದು ಬಳಕೆದಾರರಿಗೆ ಪ್ರವಾಸ ಯೋಜನೆಗಳನ್ನು ರೂಪಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತದೆ, ಮೇಲ್ಮಟ್ಟದಲ್ಲಿ ಮಾನಿಟರಿಂಗ್ ಮತ್ತು ಮೌಲ್ಯಮಾಪನ ಸೇರಿಸಲಾಗಿದೆ.


## ಸೆಟಪ್


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ಉತ್ಪಾದನಾ ಪರಿಗಣನೆಗಳು

ಪ್ರೋಟೋಟೈಪ್‌ಗಳಿಂದ ಉತ್ಪಾದನೆಗೆ AI ಏಜೆಂಟ್ಗಳನ್ನು ಕರೆದೊಯ್ಯಲು ಮೂರು ಸ್ತಂಭಗಳಿಗೆ ಸೂಕ್ಷ್ಮ ಗಮನ ಅಗತ್ಯವಿದೆ:

1. **ನಿರೀಕ್ಷಣಾ ಸಾಮರ್ಥ್ಯ** — ಏಜೆಂಟ್ ಏನು ಮಾಡುತ್ತಿದೆ, ಅದಕ್ಕೆ ಎಷ್ಟು ಸಮಯ ತೆಗೆದುಕೊಳ್ಳುತ್ತದೆ ಮತ್ತು ಅದು ಯಾವ ಉಪಕರಣಗಳನ್ನು ಕರೆದುಕೊಳ್ಳುತ್ತದೆ ಎಂಬುದರ ಬಗ್ಗೆ ನಿಮಗೆ ಗೋಚರತೆ ಅಗತ್ಯವಿದೆ. ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಲಾಗಿಂಗ್ ಇಲ್ಲದೆ ಉತ್ಪಾದನಾ ಸಮಸ್ಯೆಗಳನ್ನು ದೋಷ ಪತ್ತೆಮಾಡುವುದು ಬಹುಶಃ ಅಸಾಧ್ಯವಾಗಿದೆ.

2. **ಮೌಲ್ಯಮಾಪನ** — ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಪರಿಶೀಲನೆಗಳು ಸಮಯದೊಂದಿಗೆ ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಗಳು ನಿಖರ, ಸಂಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕವಾಗಿರುವಂತೆ ಖಚಿತಪಡಿಸುತ್ತವೆ. ಒಂದು ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ನಿರ್ಧಿಷ್ಟ ಮಾಪದಂಡಗಳ ವಿರುದ್ಧ ಪ್ರತಿಕ್ರಿಯೆಗಳಿಗೆ ಅಂಕ ನೀಡಬಹುದು.

3. **ಖರ್ಚು ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಬಳಕೆ ನೇರವಾಗಿ ವೆಚ್ಛನ್ನು ಪ್ರಭಾವಿಸುತ್ತದೆ. ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ, ಮಾದರಿ ಆಯ್ಕೆ ಮತ್ತು ಕ್ಯಾಶಿಂಗ್ ಮುಂತಾದ ತಂತ್ರಗಳು ಗುಣಮಟ್ಟಕ್ಕೆ ತ್ಯಾಗ ಮಾಡದೇ ವೆಚ್ಚವನ್ನು ನಿಯಂತ್ರಣದಲ್ಲಿ ಇಡಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


## ನೋಟಿಸಬಹುದಾದ ಏಜೆಂಟ್ ನಿರ್ಮಾಣ

ನಾವು ಪ್ರಯಾಣ ಸಾಧನಗಳನ್ನು ನಿರ್ಧರಿಸಿ ಮತ್ತು ವಿಳಂಬವನ್ನು ಮೇಲ್ವಿಚಾರಿಸಲು ಏಜೆಂಟ್ ಕರೆಗೆ ಸಮಯಮಾಪನವನ್ನು ಜೋಡಿಸುತ್ತೇವೆ. ಉತ್ಪಾದನಾತ್ಮಕ ಪರಿಸರದಲ್ಲಿ ನೀವು OpenTelemetry ಅಥವಾ ಇಂತಹ ಸಮಾನ ಟ್ರೇಸಿಂಗ್ ಬ್ಯಾನ್‌ಡ್‌ಎಂಡ್ನೊಂದಿಗೆ ಏಕೀಕರಿಸುತ್ತೀರಿ.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ಮೌಲ್ಯಮಾಪನ ಮಾದರಿಗಳು

ಸಾಮಾನ್ಯ ಉತ್ಪಾದನಾ ಮಾದರಿ ಎಂದರೆ ಎರಡನೇ ಏಜೆಂಟನ್ನು **ಮೌಲ್ಯಮಾಪಕ** ಆಗಿ ಬಳಸುವುದು. ಮೌಲ್ಯಮಾಪಕವು ಪ್ರಮುಖ ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪೂರ್ವನಿರ್ಧಿಷ್ಟ ಮಾನದಂಡಗಳಾದ ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಗಳ ವಿರುದ್ಧ ಅಂಕೆಮೌಲ್ಯನಿರ್ಣಯ ಮಾಡುತ್ತದೆ.

ಇದರಿಂದ:
- ಉತ್ತರಗಳು ಬಳಕೆದಾರರಿಗೆ ತಲುಪುವುದಕ್ಕಿಂತ ಮೊದಲು ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ತಪಾಸಣೆಗಳು
- ಪ್ರಾಂಪ್ಟ್‌ಗಳು ಅಥವಾ ಮಾದರಿಗಳು ಬದಲಾದಾಗ ರಿಗ್ರೆಶನ್ ಪತ್ತೆ
- ಕಾಲಕಾಲಾಂತರದಲ್ಲಿ ಏಜೆಂಟ್ ಕಾರ್ಯಕ್ಷಮತೆಯ ನಿರಂತರ ಮೇಲ್ವಿಚಾರಣೆ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ವೆಚ್ಚ ನಿರ್ವಹಣಾ ತಂತ್ರಗಳು

ಉತ್ಪಾದನೆಯ AI ಏಜೆಂಟ್ಗಳಿಗೆ ವೆಚ್ಚಗಳನ್ನು ನಿಯಂತ್ರಿಸುವುದು ಬಹುಮುಖ್ಯವಾಗಿದೆ. ಮುಖ್ಯ ತಂತ್ರಗಳು ಈ ಕೆಳಕಂಡವಿವೆ:

| ತಂತ್ರ | ವಿವರಣೆ |
|---|---|
| **ಪ್ರಾಂಪ್ಟ್ ಸುಧಾರಣೆ** | ಸಿಸ್ಟಮ್ ಸೂಚನೆಗಳನ್ನು ಸಂಕ್ಷಿಪ್ತವಾಗಿರಿಸಿ. ಇನ್‌ಪುಟ್ ಟೋಕನ್‌ಗಳನ್ನು ಕಡಿಮೆ ಮಾಡಲು ಅನಾವಶ್ಯಕ ಪ್ರಸಂಗವನ್ನು ತೆಗೆದುಹಾಕಿ. |
| **ಮಾದರಿ ಆಯ್ಕೆ** | ವರ್ಗೀಕರಣ ಅಥವಾ ಹೊರತೆಗೆಯುವಿಕೆಯಂತಹ ಸರಳ ಕಾರ್ಯಗಳಿಗೆ ಚಿಕ್ಕದು, ಕಡಿಮೆ ವೆಚ್ಚದ ಮಾದರಿಗಳನ್ನು (ಉದಾ. GPT-4o-mini) ಬಳಸಿ, ಮತ್ತು ಸಂಕೀರ್ಣ ನಿರ್ಣಯಕ್ಕಾಗಿ ದೊಡ್ಡ ಮಾದರಿಗಳನ್ನು ಕಾಯ್ದಿರಿಸಿ. |
| **ಕ್ಯಾಶಿಂಗ್** | ಉಪಕರಣದ ಫಲಿತಾಂಶಗಳು ಮತ್ತು频繁ವಾಗಿ ಕೇಳಲಾಗುವ ಪ್ರಶ್ನೆಗಳನ್ನು ಕ್ಯಾಶ್ ಮಾಡಿ, ಅನವಶ್ಯಕ API ಕರೆಗಳನ್ನು ತಪ್ಪಿಸಲು. |
| **ಟೋಕನ್ ಬಜೆಟ್‌ಗಳು** | ಅನಿರೀಕ್ಷಿತವಾಗಿ ಉದ್ದವಾದ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ತಡೆಯಲು `max_tokens` ಮಿತಿಗಳನ್ನು ನಿಗದಿಮಾಡಿ. |
| **ಬ್ಯಾಚಿಂಗ್** | ಸಾಧ್ಯವಾದಲ್ಲಿ, ಹಲವಾರು ಬಳಕೆದಾರ ಪ್ರಶ್ನೆಗಳನ್ನು ಒಟ್ಟಿಗೆ ಒಂದು API ಕರೆಗೆ ಗುಂಪು ಮಾಡಿ. |

ವಾಸ್ತವದಲ್ಲಿ, ಸ್ತರರೀತಿಯ ವಿಧಾನ ಉತ್ತಮವಾಗಿ ಕೆಲಸ ಮಾಡುತ್ತದೆ: ಸರಳ ವಿನಂತಿಗಳನ್ನು ವೇಗದ, ಕಡಿಮೆ ವೆಚ್ಚದ ಮಾದರಿಗೆ ಮಾರ್ಗನಿರ್ದೇಶಿಸಿ ಮತ್ತು ಕೇವಲ ಸಂಕೀರ್ಣ ಪ್ರಶ್ನೆಗಳನ್ನು ಹೆಚ್ಚು ಸಾಮರ್ಥ್ಯದ ಮಾದರಿಗೆ ಏರಿಸಿ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಳಗಿನುದನ್ನು ಹೇಗೆ ಮಾಡುವುದನ್ನು ಕಲಿತಿರಿ:

1. **ನಿರೀಕ್ಷಣೀಯತೆಯನ್ನು ಸೇರಿಸು** ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯಮಾಪನ ಮತ್ತು ಲಾಗಿಂಗ್‌ನೊಂದಿಗೆ, ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಮಾನಿಟರಿಂಗ್‌ಗಾಗಿ ನೆಲಿಯನ್ನು ಕಲಿಸಿದಂತೆ.
2. **ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಮೌಲ್ಯಮಾಪನ ಮಾಡುವುದು** ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಒಂದು ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ಉಪಯೋಗಿಸಿ, ಇದು ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯನ್ನು ಅಂಕ ನೀಡುತ್ತದೆ.
3. **ವೆಚ್ಚವನ್ನು ನಿರ್ವಹಣೆ ಮಾಡಿ** ಪ್ರಾಂಪ್ಟ್ ಆప్టಿಮೈಸೇಶನ್, ಮಾದರಿ ಆಯ್ಕೆ, ಕ್ಯಾಶಿಂಗ್, ಮತ್ತು ಟೋಕನ್ ಬಜೆಟ್‌ಗಳ ಮೂಲಕ.

ಈ ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು ನಿಮ್ಮ AI ಏಜೆಂಟ್‌ಗಳು ವಿಶ್ವಸನೀಯವಾಗಿರುವಂತೆ, ಅಳೆಯಬಹುದಾಗಿರುವಂತೆ ಮತ್ತು ಪ್ರಮಾಣದಲ್ಲಿ ವೆಚ್ಚ ಪರಿಣಾಮಕಾರಿಯಾಗಿರುವಂತೆ ಖಚಿತಪಡಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ನಿರಾಕರಣೆ:
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಮೂಲಕ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಗೆ ಪ್ರಯತ್ನಿಸಿದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅನಿಖರತೆಗಳು ಇರಬಹುದಾಗಿದೆ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿನ ದಸ್ತಾವೇಜನ್ನು ಅಧಿಕೃತ ಮೂಲವಾಗಿ ಪರಿಗಣಿಸಬೇಕು. ಗಂಭೀರ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆದಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗ್ರಹಣೆಗಳು ಅಥವಾ ತಪ್ಪಾಗಿ ವಿವರಣೆಗಳಿಂದ ಉಂಟಾಗುವ ಪರಿಣಾಮಗಳಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
